In [ ]:
# Imports
import sys
sys.path.append("Program")

from backtest import *
import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import modify_current_date, get_df
import matplotlib.pyplot as plt
import multiprocessing
import numpy as np
import pandas as pd
from pandas import ExcelWriter as EW
from plot import *
from technicals import *
from tqdm import tqdm

# Start of the program
start = dt.datetime.now()

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "HKEX", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Modify the current date
current_date = modify_current_date(start, index_name)

In [ ]:
def date_to_ff_symbol(date_str):
    """
    Convert a date string (YYYY-MM-DD) to the corresponding Fed Funds futures symbol.

    Parameters:
    - date_str (str): Date in "YYYY-MM-DD" format.

    Returns:
    - str: Fed Funds futures symbol for the given date.
    """

    monthly_futures_symbols = {
        "01": "F", "02": "G", "03": "H", "04": "J", "05": "K", "06": "M",
        "07": "N", "08": "Q", "09": "U", "10": "V", "11": "X", "12": "Z"
    }
    year, month, _ = date_str.split("-")
    return f"ZQ{monthly_futures_symbols[month]}{year[-2:]}.CBT"

def get_ff_contract_price(ff_symbol, current_date, price_df=None):
    """
    Retrieve the Fed Funds contract price for a given symbol and date.

    Parameters:
    - ff_symbol (str): Fed Funds futures symbol.
    - current_date (str): Current date in 'YYYY-MM-DD' format.
    - price_df (pd.DataFrame, optional): DataFrame containing price data.

    Returns:
    - float or None: Last price of the contract, or None if not found.
    """
    try:
        if price_df is not None:
            mask = price_df["Contract"].str.contains(ff_symbol.split(".")[0])
            return price_df.loc[mask, "Last"].values[0]
        return get_df(ff_symbol, current_date)["Close"].iloc[0]
    except Exception as e:
        print(f"Could not update price data for {ff_symbol}: {e}.")
        return None

In [ ]:
# List of FOMC meeting dates
fomc_dates = [
    "2025-03-19", "2025-05-07", "2025-06-18", "2025-07-30", "2025-09-17",
    "2025-10-29", "2025-12-10", "2026-01-28", "2026-03-18", "2026-04-29",
    "2026-06-17", "2026-07-29", "2026-09-16", "2026-10-28", "2026-12-09"
]

# Define the folder containing price data
price_data_folder = "Price data"

# List all files in the folder that start with the specified prefix
files = [file for file in os.listdir(price_data_folder) if file.startswith("30-day-fed-funds-prices-intraday")]

# Read the CSV file into a DataFrame
price_df = pd.read_csv(os.path.join(price_data_folder, files[0]))

In [ ]:
# Create a date range with monthly frequency
date_range = pd.date_range(start="2025-05-01", end="2026-12-01", freq="MS")

# Initialise an empty dataframe with the date range as the index
df = pd.DataFrame(index=date_range.strftime("%Y %b"))
df.index.name = "Month"

# Get the start date of each month
df["Start Date"] = date_range

# Add a column called "FOMC Date" and input the date if fomc_dates exist
df["FOMC Date"] = df.index.map(lambda x: next((date for date in fomc_dates if pd.to_datetime(date).strftime("%Y %b") == x), None))

# Apply the date_to_ff_symbol function to each element in the "Start Date" column
df["FF Symbol"] = df["Start Date"].apply(lambda x: date_to_ff_symbol(x.strftime("%Y-%m-%d")))

# Get the FF contract price and calculate EFFR Avg
df["FF Contract Price"] = df["FF Symbol"].apply(lambda x: get_ff_contract_price(x, current_date, price_df=price_df))
df["EFFR Avg"] = 100 - df["FF Contract Price"]

# Fill out the EFFR End and EFFR Start for the months immediately preceding and following the non-FOMC meeting month
for i in range(len(df)):
    if not df.loc[df.index[i], "FOMC Date"]:
        # Handle edge cases for the first and last rows
        df.loc[df.index[i], "EFFR Start"] = df.loc[df.index[i], "EFFR Avg"]
        df.loc[df.index[i], "EFFR End"] = df.loc[df.index[i], "EFFR Avg"]
        if i not in [0, -1]:
            df.loc[df.index[i - 1], "EFFR End"] = df.loc[df.index[i], "EFFR Start"]
            df.loc[df.index[i + 1], "EFFR Start"] = df.loc[df.index[i], "EFFR End"]
            
# Calculate the number of days before and after the FOMC meeting
df["N"] = df.apply(lambda row: (pd.to_datetime(row["FOMC Date"]) - row["Start Date"]).days + 1 if pd.notna(row["FOMC Date"]) else None, axis=1)
df["M"] = df.apply(lambda row: (row["Start Date"] + pd.offsets.MonthEnd(0)).day - pd.to_datetime(row["FOMC Date"]).day if pd.notna(row["FOMC Date"]) else None, axis=1)

# First for loop to update EFFR End and EFFR Start if one of them is NaN
for i in range(1, len(df) - 1):
    if df.loc[df.index[i], "FOMC Date"]:
        N = df.loc[df.index[i], "N"]
        M = df.loc[df.index[i], "M"]
        effr_avg = df.loc[df.index[i], "EFFR Avg"]
        effr_start = df.loc[df.index[i], "EFFR Start"]
        effr_end = df.loc[df.index[i], "EFFR End"]
        if pd.isna(effr_end) and not pd.isna(effr_start):
            df.loc[df.index[i], "EFFR End"] = (effr_avg - N / (N + M) * effr_start) / (M / (N + M))
        elif pd.isna(effr_start) and not pd.isna(effr_end):
            df.loc[df.index[i], "EFFR Start"] = (effr_avg - M / (N + M) * effr_end) / (N / (N + M))

# Second for loop to update EFFR Start and EFFR End if both are NaN
for i in range(1, len(df) - 1):
    if df.loc[df.index[i], "FOMC Date"]:
        effr_start_next = df.loc[df.index[i + 1], "EFFR Start"]
        effr_end_prev = df.loc[df.index[i - 1], "EFFR End"]
        effr_start = df.loc[df.index[i], "EFFR Start"]
        effr_end = df.loc[df.index[i], "EFFR End"]
        if pd.isna(effr_start) and pd.isna(effr_end):
            df.loc[df.index[i], "EFFR Start"] = effr_end_prev
            df.loc[df.index[i], "EFFR End"] = effr_start_next

# Calculate the change in EFFR
df["EFFR Change"] = df["EFFR End"] - df["EFFR Start"]

# Calculate the equivalent number of 25bp change
df["Number of 25bp Change"] = df["EFFR Change"] / 0.25

# Break down the number of expected 25bp changes into whole integers and remaining decimals
df["Characteristic"] = df["Number of 25bp Change"].apply(lambda x: np.floor(x) if x >= 0 else np.ceil(x))
df["Mentissa"] = df["Number of 25bp Change"] - df["Characteristic"]

# Calculate the minimum and maximum EFFR change
df["EFFR Min Change"] = 0.25 * df["Characteristic"]
df["EFFR Max Change"] = df["EFFR Min Change"] + df["Mentissa"].apply(lambda x: 0.25 * np.sign(x))

# Calculate the probabilities for the minimum and maximum EFFR change
df["Probability Min"] = df["Mentissa"].apply(lambda x: 1 - np.abs(x))
df["Probability Max"] = df["Mentissa"].apply(np.abs)

# Initialise an empty dictionary to store FOMC probabilities
fomc_dict = {}

# Get the indices of rows
indices = df[df["Probability Min"].notna()].index.tolist()

# Iterate over the indices
for i in range(len(indices)):
    month = indices[i]
    prob_dict = {}
    
    # For the first FOMC meeting
    if i == 0:
        effr_min_chg = df.loc[month, "EFFR Min Change"]
        effr_max_chg = df.loc[month, "EFFR Max Change"]
        prob_min = df.loc[month, "Probability Min"]
        prob_max = df.loc[month, "Probability Max"]
        
        # Store the probabilities for the first meeting
        prob_dict[f"{effr_min_chg}"] = prob_min
        prob_dict[f"{effr_max_chg}"] = prob_max
        fomc_dict[month] = prob_dict
    
    # For subsequent FOMC meetings
    elif i >= 1:
        month_prev = indices[i - 1]
        effr_min_chg = df.loc[month, "EFFR Min Change"]
        effr_max_chg = df.loc[month, "EFFR Max Change"]
        prob_min = df.loc[month, "Probability Min"]
        prob_max = df.loc[month, "Probability Max"]
        prob_dict_prev = fomc_dict[month_prev]
        
        # Update the probabilities based on the previous meeting
        for chg_prev, prob_prev in prob_dict_prev.items():
            min_chg_new = np.float64(chg_prev) + effr_min_chg
            max_chg_new = np.float64(chg_prev) + effr_max_chg
            prob_dict[f"{min_chg_new}"] = prob_dict.get(f"{min_chg_new}", 0) + prob_prev * prob_min
            prob_dict[f"{max_chg_new}"] = prob_dict.get(f"{max_chg_new}", 0) + prob_prev * prob_max
        
        # Store the updated probabilities
        fomc_dict[month] = prob_dict

In [ ]:
month = "2026 Jan"
current_date = "2025-05-04"

# Create a figure
plt.figure(figsize=(10, 6))

# Extract the content from the dictionary
fomc_data = fomc_dict[month]

# Extract the EFFR changes and probabilities from the dictionary
effr_chgs = [float(effr_chg) for effr_chg in fomc_data.keys()]
probs = [prob * 100 for prob in fomc_data.values()] # Convert probabilities to percentages

# Plot the histogram
bars = plt.bar(effr_chgs, probs, width=0.2, align="center")

# Change the color of the bar where EFFR change is less than or equal to -0.5 to red
for bar, effr_chg in zip(bars, effr_chgs):
    if effr_chg <= -0.5:
        bar.set_color("red")

# Add text labels on each bar
for bar, prob in zip(bars, probs):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{prob:.2f}%", ha="center", va="top", fontsize=10, color="black")

# Calculate the probability where EFFR change is equal or smaller than -50bp
prob_leq50 = sum(prob for effr_chg, prob in fomc_data.items() if float(effr_chg) <= -0.5) * 100

# Calculate the probability where EFFR change is equal or smaller than -25bp
prob_leq25 = sum(prob for effr_chg, prob in fomc_data.items() if float(effr_chg) <= -0.25) * 100

# Set the labels and title
plt.xlabel("EFFR Change (%)")
plt.ylabel("Probability (%)")
plt.title(f"EFFR Probabilities for {month} (as of {current_date})")

# Add the text to the plot
plt.text(-2, max(probs) * 0.8, fr"$P(\Delta EFFR \leq -0.5\%) = {prob_leq50:.2f}\%$", fontsize=12)
plt.text(-2, max(probs) * 0.75, fr"$P(\Delta EFFR \leq -0.25\%) = {prob_leq25:.2f}\%$", fontsize=12)

# Adjust the spacing
plt.tight_layout()

# Save the plot
filename = f"Result/Figure/effr{month.replace(' ', '').lower()}.png"
plt.savefig(filename, dpi=300)

# Show the plot
plt.show()